In [1]:
# %pip install pennylane pennylane-qiskit
# %pip install torch scikit-learn numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:

# Build a Simple QNN Circuit (4 Qubits)
import pennylane as qml
from pennylane import numpy as np
import torch
import torch.nn as nn

n_qubits = 4
n_layers = 2

dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def qnn_layer(inputs, weights):
    # Encode classical data
    for i in range(n_qubits):
        qml.RX(inputs[i], wires=i)

    # Variational layers
    qml.templates.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


In [10]:
#Wrap it as a Torch Model
class HybridQNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.q_weights = nn.Parameter(
            0.01 * torch.randn(n_layers, n_qubits, 3)
        )
        self.fc = nn.Linear(n_qubits, 1)

    def forward(self, x):
        q_out = []
        for sample in x:
            q = qnn_layer(sample, self.q_weights)
            q = torch.stack(q)
            q_out.append(q)
        q_out = torch.stack(q_out).float()
        return self.fc(q_out)


In [11]:
#Prepare Dataset (Tiny!)
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = make_moons(n_samples=300, noise=0.1)
scaler = StandardScaler()
X = scaler.fit_transform(X)

# pad to 4 features (for 4 qubits)
import numpy as np
X = np.hstack([X, np.zeros((X.shape[0], 2))])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)


In [ ]:
#Train the QNN
model = HybridQNN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.BCEWithLogitsLoss()

epochs = 30

for ep in range(epochs):
    optimizer.zero_grad()
    y_pred = model(X_train).squeeze()
    loss = loss_fn(y_pred, y_train)
    loss.backward()
    optimizer.step()

    print(f"Epoch {ep+1}: Loss = {loss.item():.4f}")

Epoch 1: Loss = 0.6983
Epoch 2: Loss = 0.6974
Epoch 3: Loss = 0.6969
Epoch 4: Loss = 0.6965
Epoch 5: Loss = 0.6961
Epoch 6: Loss = 0.6957
Epoch 7: Loss = 0.6953
Epoch 8: Loss = 0.6949
Epoch 9: Loss = 0.6945
Epoch 10: Loss = 0.6941
Epoch 11: Loss = 0.6937
Epoch 12: Loss = 0.6934
Epoch 13: Loss = 0.6931
Epoch 14: Loss = 0.6928
Epoch 15: Loss = 0.6926
Epoch 16: Loss = 0.6924
Epoch 17: Loss = 0.6922
Epoch 18: Loss = 0.6921
Epoch 19: Loss = 0.6919
Epoch 20: Loss = 0.6918
Epoch 21: Loss = 0.6917
Epoch 22: Loss = 0.6915
Epoch 23: Loss = 0.6913
Epoch 24: Loss = 0.6911
Epoch 25: Loss = 0.6908
Epoch 26: Loss = 0.6904
Epoch 27: Loss = 0.6900
Epoch 28: Loss = 0.6895
Epoch 29: Loss = 0.6889
Epoch 30: Loss = 0.6882


In [ ]:
# Evaluate
with torch.no_grad():
    preds = torch.sigmoid(model(X_test)).squeeze()
    preds = (preds > 0.5).float()

acc = (preds == torch.tensor(y_test, dtype=torch.float32)).float().mean()
print("Accuracy:", acc.item())

Accuracy: 0.4333333373069763
